In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/test'

In [3]:
def createdf(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir,label)):
            image_paths.append(os.path.join(dir,label,imagename))
            labels.append(label)
        print(label,"completed")
    return image_paths,labels

In [4]:
train = pd.DataFrame()
train['image'],train['label'] = createdf(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [5]:
test = pd.DataFrame()
test['image'],test['label'] = createdf(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [6]:
from tqdm import tqdm
from keras.preprocessing.image import load_img

In [7]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image,grayscale=True)
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features),48,48,1)
    return features

In [8]:
train_features = extract_features(train['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]c:\Users\Yug Sondagar\MLProjects-2\FACE_EMOTION_DETECTION\venv\Lib\site-packages\keras\src\utils\image_utils.py:409: UserWarning: grayscale is deprecated. Please use color_mode = "grayscale"
  warnings.warn(
100%|██████████| 28821/28821 [06:57<00:00, 69.00it/s] 


In [9]:
test_features = extract_features(test['image'])

100%|██████████| 7066/7066 [01:49<00:00, 64.67it/s]


In [10]:
X_train = train_features/255
X_test = test_features/255

In [11]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [12]:
y_train = le.fit_transform(train['label'])
y_test = le.transform(test['label'])

In [13]:
from keras.utils import to_categorical

In [14]:
y_train = to_categorical(y_train,num_classes=7)
y_test = to_categorical(y_test,num_classes=7)

In [15]:
from keras.models import Sequential
from keras.layers import Conv2D,Dense,Dropout,Flatten,MaxPooling2D

In [16]:
model = Sequential()
#Convolutional layer
model.add(Conv2D(128,kernel_size=(3,3),activation='relu',input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())

model.add(Dense(512,activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256,activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(7,activation='softmax'))

In [17]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [18]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_accuracy', 
                           patience=5, 
                           restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    batch_size=128,
    validation_data=(X_test, y_test),
    epochs=30,
    callbacks=[early_stop]
)


Epoch 1/30


226/226 [==============================] - 433s 2s/step - loss: 1.8220 - accuracy: 0.2449 - val_loss: 1.7976 - val_accuracy: 0.2615
Epoch 2/30
226/226 [==============================] - 436s 2s/step - loss: 1.7623 - accuracy: 0.2707 - val_loss: 1.6785 - val_accuracy: 0.3254
Epoch 3/30
226/226 [==============================] - 386s 2s/step - loss: 1.6400 - accuracy: 0.3445 - val_loss: 1.4940 - val_accuracy: 0.4362
Epoch 4/30
226/226 [==============================] - 324s 1s/step - loss: 1.5124 - accuracy: 0.4091 - val_loss: 1.3792 - val_accuracy: 0.4641
Epoch 5/30
226/226 [==============================] - 303s 1s/step - loss: 1.4513 - accuracy: 0.4371 - val_loss: 1.3157 - val_accuracy: 0.4973
Epoch 6/30
226/226 [==============================] - 292s 1s/step - loss: 1.3908 - accuracy: 0.4615 - val_loss: 1.2648 - val_accuracy: 0.5075
Epoch 7/30
226/226 [==============================] - 301s 1s/step - loss: 1.3541 - accuracy: 0.4791 - val_loss: 1.2366 - val_accuracy: 0.52

In [19]:
model.save('emotion_detector.h5')

c:\Users\Yug Sondagar\MLProjects-2\FACE_EMOTION_DETECTION\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
